# Unified Training Pipeline with Dynamic Training

This notebook integrates data preprocessing and training of a ResNet for both binary and multiclass motor fault classification. Each epoch a new training dataset is generated

In [ ]:
# Logging
import comet_ml
import yaml
import datetime

with open("../cfg.yaml", "r") as f:
    config = yaml.safe_load(f)

api_key = config['API_KEY']
project_name = config['PROJECT_NAME']
workspace = config['WORKSPACE']

experiment = comet_ml.start(api_key=api_key, 
                        project_name=project_name, 
                        workspace=workspace)

current_time = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(current_time)
experiment.set_name(f"Run: {current_time}")
experiment.log_code()

In [ ]:
import numpy as np
import random

from src.data_pipeline import (
    generate_synthetic_peak, process_file, insert_synthetic_peaks, create_dataloaders, create_dataset, normalize_segment
)
from src.electrical_signature_frequencies import ANOMALY_FREQS
from src.models import ResNet, ResidualBlock
from src.train import train_model
from src.inference import inference_resnet_model
from src.evaluation import calculate_metrics
from src.anomaly_injector import GaussianPeakInjector, NoiseInjector, CompositeAnomalyInjector

from sklearn.metrics import classification_report
import seaborn as sns
import matplotlib.pyplot as plt


import torch
from torch.utils.data import TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set device 
device: str = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
experiment.log_parameter("device", device)
device = torch.device(device)
print("Using device:", device)

## 1. Load Engine Configuration and Set Parameters

In [ ]:
# Load training parameters from train.yml
with open("../training_configs/train.yml", "r") as f:
    train_params = yaml.safe_load(f)

# Processing Parameters
f_sampling = train_params["processing_parameters"]["f_sampling"]
shift = train_params["processing_parameters"]["shift"]
segment_length = train_params["processing_parameters"]["segment_length"]
cutoff_freq = train_params["processing_parameters"]["cutoff_freq"]
peak_type = train_params["processing_parameters"]["peak_type"]
peak_segment = train_params["processing_parameters"]["peak_segment"]
amplitude_range = tuple(train_params["processing_parameters"]["amplitude_range"])
sigma_range = tuple(train_params["processing_parameters"]["sigma_range"])
noise_factor = train_params["processing_parameters"]["noise_factor"]

# Training Parameters
num_epochs_binary = train_params["training_parameters"]["num_epochs_binary"]
num_epochs_multi = train_params["training_parameters"]["num_epochs_multi"]

# Data Parameters
batch_size = train_params["data_parameters"]["batch_size"]
file_label_map = train_params["data_parameters"]["file_label_map"]
fault_types_available = train_params["data_parameters"]["fault_types_available"]

# Dataset Parameters
normalization_method = train_params["dataset_parameters"]["normalization_method"]

# Load Engine Configuration
with open('../engine_configs/LIMAN.yml', 'r') as f:
    engine_config = yaml.safe_load(f)

experiment.log_parameters({
    "f_sampling": f_sampling,
    "shift": shift,
    "segment_length": segment_length,
    "peak_segment": peak_segment,
    "cutoff_freq": cutoff_freq,
    "file_label_map": file_label_map,
    "fault_types_available": fault_types_available,
    "peak_type": peak_type,
    "num_epochs_binary": num_epochs_binary,
    "num_epochs_multi": num_epochs_multi,
    "batch_size": batch_size,
    "normalization_method": normalization_method,
    "amplitude_range": amplitude_range,
    "sigma_range": sigma_range,
    "noise_factor": noise_factor
})

MCSA_cfg = {
    'rotor bar defect': {
        'engine_config': engine_config,
        'n_range': range(1, 4)
        },
    'inter-turn short circuits': {
        'engine_config': engine_config,
        'k_range': range(1, 4, 2), 
        'm_range': range(0, 1)
        }
}

fault_freqs = {
    fault_type: ANOMALY_FREQS[fault_type](**MCSA_cfg[fault_type])
    for fault_type in fault_types_available
}
experiment.log_parameters(fault_freqs)

In [ ]:
# Instantiate the Gaussian injector
gaussian_injector = GaussianPeakInjector(
    peak_segment=peak_segment,
    amplitude_range=amplitude_range,
    sigma_range=sigma_range
)

# Instantiate the Noise injector
noise_injector = NoiseInjector(noise_factor=noise_factor)

# Combine the injectors
composite_injector = CompositeAnomalyInjector({
    'peak-anomaly': gaussian_injector,
    'noise': noise_injector
})

In [ ]:
# Saving spectrums before and after peak injection

sample_file = None
for fp, lbl in file_label_map.items():
    if lbl == 'normal':
        sample_file = fp
        break

if sample_file is not None:
    try:
        segs, labels, freqs = process_file(sample_file, 'normal',
                                            f_sampling=f_sampling, cutoff_freq=cutoff_freq,
                                            segment_length=segment_length, step=shift,
                                            apply_window=False, db=True,
                                            synthetic_fault_fraction=0.0, fault_types=None, anomaly_injector=composite_injector)
    except NameError:
        pass

    if segs is not None and len(segs) > 0:
        sample_segment = segs[0]
        for fault_type in fault_types_available:
            injected_segment = composite_injector.inject(sample_segment, freqs, fault_freqs[fault_type])
            difference = injected_segment - sample_segment

            fig, axs = plt.subplots(1, 3, figsize=(18, 5))
            axs[0].plot(freqs, sample_segment, color='blue')
            axs[0].set_title("Before Processing")
            axs[0].set_xlabel("Frequency")
            axs[0].set_ylabel("Magnitude (dB)")
            axs[0].grid(True)
            
            axs[1].plot(freqs, injected_segment, color='orange')
            axs[1].set_title(f"After Processing ({fault_type})")
            axs[1].set_xlabel("Frequency")
            axs[1].grid(True)
            
            axs[2].plot(freqs, difference, color='green')
            axs[2].set_title("Difference (After - Before)")
            axs[2].set_xlabel("Frequency")
            axs[2].grid(True)
            
            fig.suptitle(f"Spectrum Comparison for {fault_type}")
            experiment.log_figure(figure=fig, figure_name=f"spectrum_comparison_{fault_type}.png")

## 2. Train the ResNet Binary Classifier

In [ ]:
import numpy as np
from torch.utils.data import Dataset
from typing import Any, Dict, List, Optional, Tuple

from src.data_pipeline import normalize_segment

class DynamicDataset(Dataset):
    """
    A PyTorch Dataset for dynamic anomaly injection during training, where each segment
    can be assigned any label (normal or one of the available fault types) with equal probability.
    
    This forces the model to learn features indicative of anomalies rather than memorizing
    specific segments. Each time a sample is fetched, an anomaly can be injected according
    to the chosen label and an optional noise injector can be applied.
    
    Attributes:
        raw_segments (np.ndarray): Array of raw training segments (not preprocessed).
                                   Shape typically [num_segments, segment_length].
        raw_labels (np.ndarray): Array of corresponding labels for each segment (unused for
                                 the dynamic labeling, but retained for reference).
        freqs (np.ndarray): Array of frequency bins for each segment (e.g., from an FFT).
        engine_config (Dict[str, Any]): Configuration parameters for the engine, used by injectors.
        mode (str): Either "binary" or "multiclass". In "binary" mode, the returned label is 0 for
                    normal and 1 for any fault. In "multiclass" mode, the returned label is the
                    chosen fault type (string) or "normal".
        fault_types_available (List[str], optional): List of fault type names that can be injected.
        normalization_method (str, optional): Name of the normalization method to apply on the fly.
                                              Examples: "z-score", "min-max". If None, no normalization.
        anomaly_injector (Any, optional): An injector (e.g., CompositeAnomalyInjector) responsible
                                          for inserting synthetic anomalies and/or noise.
    """

    def __init__(
        self,
        raw_segments: np.ndarray,
        raw_labels: np.ndarray,
        freqs: np.ndarray,
        engine_config: Dict[str, Any],
        mode: str = "binary",
        fault_types_available: Optional[List[str]] = None,
        normalization_method: Optional[str] = 'min-max',
        anomaly_injector: Optional[Any] = None,
        label_encoder: Optional[LabelEncoder] = None
    ) -> None:
        """
        Initializes the DynamicDataset with raw data, optional anomaly injectors, and
        user-specified modes.

        Args:
            raw_segments (np.ndarray): Raw training segments (not preprocessed).
                                       Shape [num_segments, segment_length].
            raw_labels (np.ndarray): Corresponding labels for each segment. 
                                     (Not used for label selection in this dynamic dataset,
                                     but kept for reference or debugging.)
            freqs (np.ndarray): FFT frequency bins (same length as each segment).
            engine_config (Dict[str, Any]): Configuration dictionary for the engine, used
                                            by anomaly injectors to determine fault frequencies.
            mode (str): "binary" or "multiclass".
                        - In "binary" mode, the returned label is 0 for normal and 1 for any fault.
                        - In "multiclass" mode, the returned label is the string fault type or "normal".
            fault_types_available (List[str], optional): List of fault types that can be injected
                                                        (e.g., ["inter-turn short circuits", "rotor bar defect"]).
            normalization_method (str, optional): Normalization method applied on-the-fly.
                                                 If None, no normalization is applied.
            anomaly_injector (Any, optional): An injector instance to perform synthetic anomaly
                                              injection. Must have an `inject()` method accepting
                                              (segment, freqs, fault_type, injector_keys).
        """
        super().__init__()
        self.raw_segments = raw_segments
        self.raw_labels = raw_labels
        self.freqs = freqs
        self.engine_config = engine_config
        self.mode = mode
        self.fault_types_available = fault_types_available or []
        self.normalization_method = normalization_method
        self.anomaly_injector = anomaly_injector
        self.label_encoder = label_encoder

        # Build the list of possible classes, including "normal"
        self.all_classes = set(self.fault_types_available)
        self.all_classes.add("normal")
        self.all_classes = list(self.all_classes)  # Convert to list for uniform sampling

    def __len__(self) -> int:
        """
        Returns:
            int: The total number of segments in the dataset.
        """
        return len(self.raw_segments)

    def __getitem__(self, idx: int) -> Tuple[np.ndarray, Any]:
        """
        Retrieves a single sample from the dataset, dynamically assigning a label
        and injecting anomalies as appropriate.

        Workflow:
            1. Select a label from self.all_classes with uniform probability.
            2. If the label is "normal", no fault injection is applied.
               Otherwise, inject the corresponding fault (via 'peak-anomaly' key).
            3. Optionally inject noise if the anomaly_injector has a 'noise' key.
            4. Normalize the segment if a normalization_method is specified.
            5. Return the segment (with shape [1, segment_length]) and its label.

        Args:
            idx (int): The index of the segment to retrieve.

        Returns:
            Tuple[np.ndarray, Any]: A tuple (segment, label) where:
                - segment is a float32 numpy array of shape [1, segment_length].
                - label is either:
                    * 0 (normal) or 1 (fault) in "binary" mode, or
                    * a string among {"normal", <fault_types>} in "multiclass" mode.
        """
        # Copy the segment to avoid in-place modifications
        seg = self.raw_segments[idx].copy()

        # 1) Select a label with uniform probability from "normal" + all faults
        chosen_label_str = np.random.choice(self.all_classes)

        # 2) If the chosen label is not "normal", inject the fault anomaly
        is_normal = (chosen_label_str == "normal")
        if not is_normal and self.anomaly_injector is not None:
            seg = self.anomaly_injector.inject(
                segment=seg,
                fft_freqs=self.freqs,
                fault_freqs=fault_freqs[chosen_label_str],
                injector_keys=['peak-anomaly']
            )

        # 3) Assign the final label based on mode
        if self.mode == "binary":
            label = 0 if is_normal else 1
        else:
            label = self.label_encoder.transform([chosen_label_str])[0]

        # 4) Inject noise if available
        if self.anomaly_injector is not None and 'noise' in self.anomaly_injector.injectors:
            seg = self.anomaly_injector.inject(
                segment=seg,
                fft_freqs=self.freqs,
                fault_freqs=None,
                injector_keys=['noise']
            )

        # 5) Normalize the segment (if a method is specified)
        if self.normalization_method is not None:
            seg, _ = normalize_segment(seg, method=self.normalization_method)

        # Convert to float32 and add channel dimension: [1, segment_length]
        seg = seg.astype(np.float32)
        seg = np.expand_dims(seg, axis=0)

        return seg, label

In [ ]:
def create_datasets(
    file_label_map: Dict[str, Any],
    engine_config: Dict[str, Any],
    mode: str = "binary",
    dynamic_training: bool = True,
    f_sampling: int = 10000,
    cutoff_freq: int = 250,
    segment_length: int = 10000,
    step: int = 20,
    apply_window: bool = False,
    db: bool = True,
    fault_types_available: Optional[List[str]] = None,
    normalization_method: Optional[str] = 'min-max',
    test_size: float = 0.3,
    val_size: float = 0.2,
    seed: int = 42,
    anomaly_injector: Optional[Any] = None,
    include_real_anomalies_in_training: bool = False
) -> Dict[str, Any]:
    """
    Creates unified PyTorch datasets for training, validation, and testing.

    The function processes raw files (using process_file) and splits the data into train/val/test.
    - For files labeled "normal":
        - In static mode (dynamic_training=False), a fraction of training segments is pre-modified 
          using the anomaly_injector (if provided) and then normalized.
        - In dynamic mode, raw segments are kept intact and anomalies are injected on the fly.
    - For fault files, the data is always processed statically.

    Returns:
        dict: A dictionary with keys "train", "val", and "test" containing PyTorch Dataset objects.
              For multiclass mode, a "label_encoder" is also included.
    """
    # Determine synthetic injection fraction.
    if mode == "binary":
        synthetic_fraction = 0.5
    elif mode == "multiclass":
        if not fault_types_available or len(fault_types_available) == 0:
            raise ValueError("For multiclass mode, fault_types_available must be provided and non-empty.")
        synthetic_fraction = len(fault_types_available) / (len(fault_types_available) + 1)
    else:
        raise ValueError("Mode must be either 'binary' or 'multiclass'")

    # Initialize lists for segments and labels.
    train_segments, train_labels = [], []
    val_segments, val_labels = [], []
    test_segments, test_labels = [], []
    common_freqs = None

    # Process each file (without any injection).
    for file_path, label in file_label_map.items():
        segs, labels_out, freqs = process_file(
            file_path, label,
            f_sampling=f_sampling, cutoff_freq=cutoff_freq,
            segment_length=segment_length, step=step,
            apply_window=apply_window, db=db,
            synthetic_fault_fraction=0.0,  # Baseline processing: no injection.
            fault_types=None,
            anomaly_injector=None
        )

        if common_freqs is None:
            common_freqs = freqs

        # For "normal" files, perform train/val/test split and (optionally) static anomaly injection.
        if label == "normal":
            segs_train_val, segs_test, labels_train_val, labels_test = train_test_split(
                segs, labels_out, test_size=test_size, random_state=seed
            )
            segs_train, segs_val, labels_train, labels_val = train_test_split(
                segs_train_val, labels_train_val, test_size=val_size, random_state=seed
            )

            n_train = len(segs_train)
            n_modify = int(n_train * synthetic_fraction)
            modify_indices = random.sample(range(n_train), n_modify)
            new_train_labels = []

            # Static injection if not using dynamic training.
            if not dynamic_training and anomaly_injector is not None:
                new_train_segs = []
                for i, seg in enumerate(segs_train):
                    if i in modify_indices:
                        chosen_fault = random.choice(fault_types_available)
                        mod_seg = anomaly_injector.inject(seg, freqs, chosen_fault)
                        new_train_segs.append(mod_seg)
                        new_train_labels.append(chosen_fault)
                    else:
                        new_train_segs.append(seg)
                        new_train_labels.append("normal")
                segs_train, labels_train = new_train_segs, new_train_labels
            else:
                # In dynamic training, only labels are adjusted.
                for i in range(n_train):
                    if i in modify_indices:
                        chosen_fault = random.choice(fault_types_available)
                        new_train_labels.append(chosen_fault)
                    else:
                        new_train_labels.append("normal")
                labels_train = new_train_labels

            train_segments.extend(segs_train)
            train_labels.extend(labels_train)
            # For binary mode, convert "normal" to 0.
            val_segments.extend(segs_val)
            val_labels.extend([0 if mode == "binary" and l == "normal" else l for l in labels_val])
            test_segments.extend(segs_test)
            test_labels.extend([0 if mode == "binary" and l == "normal" else l for l in labels_test])
        else:
            # For fault files, decide whether to include them in training.
            segs_train_val, segs_test, labels_train_val, labels_test = train_test_split(
                segs, labels_out, test_size=test_size, random_state=seed
            )
            segs_train, segs_val, labels_train, labels_val = train_test_split(
                segs_train_val, labels_train_val, test_size=val_size, random_state=seed
            )
            if include_real_anomalies_in_training:
                train_segments.extend(segs_train)
                train_labels.extend(labels_train if mode == "multiclass" else [1] * len(labels_train))
            else:
                val_segments.extend(segs_train)
                val_labels.extend(labels_train if mode == "multiclass" else [1] * len(labels_train))
            val_segments.extend(segs_val)
            val_labels.extend(labels_val if mode == "multiclass" else [1] * len(labels_val))
            test_segments.extend(segs_test)
            test_labels.extend(labels_test if mode == "multiclass" else [1] * len(labels_test))

    # Normalize segments. For dynamic training, training segments are normalized on the fly.
    if not dynamic_training and normalization_method:
        X_train = np.array(
            [normalize_segment(seg, method=normalization_method)[0] for seg in train_segments],
            dtype=np.float32
        )
    else:
        X_train = np.array(train_segments, dtype=np.float32)

    X_val = (
        np.array([normalize_segment(seg, method=normalization_method)[0] for seg in val_segments], dtype=np.float32)
        if normalization_method else np.array(val_segments, dtype=np.float32)
    )
    X_test = (
        np.array([normalize_segment(seg, method=normalization_method)[0] for seg in test_segments], dtype=np.float32)
        if normalization_method else np.array(test_segments, dtype=np.float32)
    )

    # Process labels and encode if needed.
    y_train_raw = np.array(train_labels)
    y_val_raw = np.array(val_labels)
    y_test_raw = np.array(test_labels)

    if mode == "binary":
        final_y_train, final_y_val, final_y_test = y_train_raw, y_val_raw, y_test_raw
        label_encoder = None
    else:
        le = LabelEncoder()
        final_y_train = le.fit_transform(y_train_raw)
        final_y_val = le.transform(y_val_raw)
        final_y_test = le.transform(y_test_raw)
        label_encoder = le

    # Wrap the training data into a dataset.
    if dynamic_training:
        train_dataset = DynamicDataset(
            raw_segments=X_train,
            raw_labels=final_y_train,
            freqs=common_freqs,
            engine_config=engine_config,
            mode=mode,
            fault_types_available=fault_types_available,
            normalization_method=normalization_method,
            anomaly_injector=anomaly_injector,
            label_encoder=label_encoder
        )
    else:
        X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1)
        # Convert "normal" to 0 in binary mode.
        if mode == 'binary':
            final_y_train = np.where(final_y_train == "normal", 0, 1)
        y_train_tensor = torch.tensor(final_y_train, dtype=torch.long)
        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

    # Wrap validation and test sets into TensorDatasets.
    X_val_tensor = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1)
    y_val_tensor = torch.tensor(final_y_val, dtype=torch.long)
    val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1)
    y_test_tensor = torch.tensor(final_y_test, dtype=torch.long)
    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    datasets = {"train": train_dataset, "val": val_dataset, "test": test_dataset}
    if mode == "multiclass":
        datasets["label_encoder"] = label_encoder

    return datasets

In [ ]:
from torch.utils.data import DataLoader

# Create datasets (with dynamic training)

datasets_binary = create_datasets(
    file_label_map=file_label_map,
    engine_config=engine_config,
    mode="binary",  # or "multiclass"
    dynamic_training=True,
    f_sampling=f_sampling,
    cutoff_freq=cutoff_freq,
    segment_length=segment_length,
    step=shift,
    apply_window=False,
    db=True,
    fault_types_available=fault_types_available,
    normalization_method=normalization_method,
    test_size=0.3,
    val_size=0.2,
    seed=SEED,
    anomaly_injector=composite_injector
)

In [ ]:
# Loaders
train_loader_binary = DataLoader(datasets_binary["train"], batch_size=batch_size, shuffle=True)
val_loader_binary   = DataLoader(datasets_binary["val"], batch_size=batch_size, shuffle=False)
test_loader_binary  = DataLoader(datasets_binary["test"], batch_size=batch_size, shuffle=False)

In [ ]:
# Initialize ResNet
model_bin = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=2, block_kwargs={'use_se': True}).to(device)
print("Training ResNet for binary classification...")
model_bin = train_model(model_bin, train_loader_binary, val_loader=val_loader_binary, device=device, num_epochs=num_epochs_binary)

In [ ]:
true_labels_bin, predictions_bin = inference_resnet_model(model_bin, test_loader_binary, device=device)

In [ ]:
# Calculate metrics 
cm_bin, acc_bin, prec_bin, rec_bin, f1_bin = calculate_metrics(true_labels_bin, predictions_bin)

In [ ]:
binary_report = classification_report(true_labels_bin, predictions_bin, target_names=['normal', 'anomalous'], output_dict=True)
binary_report_prefixed = {f"binary_{key}": value for key, value in binary_report.items()}
experiment.log_metrics(binary_report_prefixed)

In [ ]:
fig_bin, ax_bin = plt.subplots(figsize=(5,4))
sns.heatmap(cm_bin, annot=True, fmt='d', cmap='Blues',
            xticklabels=['normal','anomalous'], yticklabels=['normal','anomalous'], ax=ax_bin)
ax_bin.set_xlabel('Predicted Label')
ax_bin.set_ylabel('True Label')
ax_bin.set_title('Binary Classification - Confusion Matrix')
experiment.log_figure(figure=fig_bin, figure_name="confusion_matrix_binary.png")
plt.close(fig_bin)

## 3. Train the ResNet Multiclass Classifier

In [ ]:
datasets_multi = create_datasets(
    file_label_map=file_label_map,
    engine_config=engine_config,
    mode="multiclass",
    dynamic_training=True,
    f_sampling=f_sampling,
    cutoff_freq=cutoff_freq,
    segment_length=segment_length,
    step=shift,
    apply_window=False,
    db=True,
    fault_types_available=fault_types_available,
    normalization_method=normalization_method,
    test_size=0.3,
    val_size=0.2,
    seed=SEED,
    anomaly_injector=composite_injector
)

print("Classes:", datasets_multi['label_encoder'].classes_)

In [ ]:
# Loaders
train_loader_multi = DataLoader(datasets_multi["train"], batch_size=batch_size, shuffle=True)
val_loader_multi = DataLoader(datasets_multi["val"], batch_size=batch_size, shuffle=False)
test_loader_multi = DataLoader(datasets_multi["test"], batch_size=batch_size, shuffle=False)

In [ ]:
# Train ResNet for Multiclass Classification
num_classes_multi = len(datasets_multi['label_encoder'].classes_)
model_multi = ResNet(ResidualBlock, [2, 2, 2, 2], num_classes=num_classes_multi, block_kwargs={'use_se': True}).to(device)
print("Training ResNet for multiclass classification...")
model_multi = train_model(model_multi, train_loader_multi, val_loader=val_loader_multi, device=device, num_epochs=num_epochs_multi)

In [ ]:
true_labels_multi, predictions_multi = inference_resnet_model(model_multi, test_loader_multi, device=device)

In [ ]:
# Calculate metrics 
cm_multi, acc_multi, prec_multi, rec_multi, f1_multi = calculate_metrics(true_labels_multi, predictions_multi)

In [ ]:
multiclass_report = classification_report(true_labels_multi, predictions_multi, target_names=datasets_multi['label_encoder'].classes_, output_dict=True)
multiclass_report_prefixed = {f"multiclass_{key}": value for key, value in multiclass_report.items()}
experiment.log_metrics(multiclass_report_prefixed)

In [ ]:
fig_multi, ax_multi = plt.subplots(figsize=(5,4))
sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
            xticklabels=datasets_multi['label_encoder'].classes_,
            yticklabels=datasets_multi['label_encoder'].classes_, ax=ax_multi)
ax_multi.set_xlabel('Predicted Label')
ax_multi.set_ylabel('True Label')
ax_multi.set_title('Multiclass Classification - Confusion Matrix')
experiment.log_figure(figure=fig_multi, figure_name="confusion_matrix_multiclass.png")
plt.close(fig_multi)

In [ ]:
experiment.end()